# Energy Grid Demand & Renewable Power Forecasting with Uncertainty Quantification

## Unit 1: Machine Learning & Probability Theory — Complete Project

---

**Author:** B.Tech CSE Student | **Date:** September 2026 | **Course:** Machine Learning Unit 1

---

> This notebook demonstrates a complete ML pipeline applied to a real-world energy forecasting problem.
> Every concept is connected back to the **Unit 1 syllabus** covering Machine Learning fundamentals
> and Probability Theory.

## 1. Introduction — What is Machine Learning and Why is it Needed?

### 1.1 What is Machine Learning?

**Machine Learning (ML)** is a branch of Artificial Intelligence where computer programs learn patterns from data
without being explicitly programmed with rules.

> **Formal Definition (Tom Mitchell, 1997):**
> "A computer program is said to learn from experience E with respect to some task T and performance measure P,
> if its performance at T, as measured by P, improves with experience E."

In our project:
- **Task T:** Predict electricity demand given weather and time features
- **Experience E:** Historical hourly data of demand, temperature, solar/wind generation
- **Performance P:** Mean Absolute Error (MAE), R² Score

### 1.2 Traditional Programming vs Machine Learning

| Aspect | Traditional Programming | Machine Learning |
|--------|------------------------|-----------------|
| Input | Rules + Data | Data + Answers (labels) |
| Output | Answers | Rules (learned model) |
| Adaptability | Manual rule updates | Automatic learning from new data |
| Energy Example | IF temp > 35 THEN demand = HIGH | Learn demand = f(temp, hour, wind, ...) from 8,760 hours of data |

### 1.3 Supervised vs Unsupervised Learning

**Supervised Learning:** We have both input features (X) and the correct output (Y). The model learns the mapping $f: X \rightarrow Y$.
- *Our example:* Predict electricity demand (Y) from temperature, solar generation, hour (X)

**Unsupervised Learning:** We only have input features (X), no labels. The model discovers hidden structure.
- *Our example:* Cluster hours into operational regimes (Winter Peak, Summer Peak, Night Off-Peak, Windy)

### 1.4 Why Uncertainty Quantification Matters in Energy Forecasting

A point prediction alone ("demand will be 25,000 MW") is **dangerous** for grid operators because:

1. **Grid stability** requires balancing supply = demand every second
2. **Reserve margins** must account for forecast errors
3. **Renewable intermittency** (clouds, wind lulls) adds unpredictability

Our system provides:
- **Point Prediction:** Best estimate of demand
- **Prediction Interval:** [Lower Bound, Upper Bound] — a range where actual demand likely falls
- **Probability of High Demand:** Bayesian probability that demand exceeds a critical threshold

## 2. Problem Statement

**Given** hourly measurements of temperature, humidity, solar generation, wind generation, and other environmental
variables for a full year (8,760 hours), build a system that:

1. Predicts future electricity demand (MW)
2. Quantifies prediction uncertainty using probability distributions, variance, and quantiles
3. Identifies grid operational regimes using unsupervised learning
4. Demonstrates all Unit 1 ML and Probability Theory concepts through the energy grid context

## 3. Objectives

1. **Demonstrate ML fundamentals** through a practical energy forecasting pipeline
2. **Apply probability theory** (Bayes' Rule, conditional probability, independence) to real energy data
3. **Implement polynomial curve fitting** to model nonlinear temperature-demand relationships
4. **Build supervised regression models** comparing Linear vs Polynomial Feature-Engineered models
5. **Apply K-Means clustering** to discover latent grid operating regimes
6. **Quantify uncertainty** via residual analysis, quantiles, and prediction intervals
7. **Connect every concept** to the Unit 1 syllabus with: Concept → Formula → Code → Result → Interpretation

## 4. Dataset Description

Our dataset contains **8,760 hourly records** spanning the full year 2023 with the following variables:

| Column | Description | Unit | Type |
|--------|-------------|------|------|
| `timestamp` | Date and time of observation | YYYY-MM-DD HH:MM:SS | Datetime |
| `electricity_demand_mw` | Total grid electricity demand | MW | Continuous (Target) |
| `solar_generation_mw` | Solar photovoltaic power output | MW | Continuous |
| `wind_generation_mw` | Wind turbine power output | MW | Continuous |
| `temperature_celsius` | Ambient air temperature | °C | Continuous |
| `humidity_percent` | Relative humidity | % | Continuous |
| `wind_speed_ms` | Wind speed at hub height | m/s | Continuous |
| `cloud_cover_percent` | Sky cloud cover fraction | % | Continuous |

The dataset has realistic imperfections: **25 missing values** and **5 duplicate rows** for preprocessing demonstration.

## 5. Data Preprocessing

### Steps:
1. Load dataset with automatic column adaptation
2. Inspect shape, dtypes, first rows
3. Detect and remove duplicate rows
4. Detect and handle missing values via time-aware linear interpolation
5. Parse datetime, sort chronologically
6. Engineer temporal features (hour, day_of_week, month, is_weekend, season)
7. Compute derived variables (renewable_generation_mw, net_load_mw)
8. Chronological train/test split (80/20)

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Ensure imports work from notebook directory
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
%matplotlib inline

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("All libraries imported successfully!")

In [ ]:
# Load dataset with automatic column adaptation
from data.data_loader import load_energy_data
from src.preprocessing import inspect_dataset, clean_and_preprocess, split_train_test

df_raw, _ = load_energy_data("data/energy_grid_dataset.csv")

# Display first 5 rows
print("\n=== First 5 Rows ===")
print(df_raw.head())

# Inspect dataset health
inspection = inspect_dataset(df_raw)
print(f"\n=== Dataset Shape: {inspection['shape']} ===")
print(f"\n=== Data Types ===")
for col, dtype in inspection['dtypes'].items():
    print(f"  {col}: {dtype}")
print(f"\n=== Missing Values ===")
for col, count in inspection['missing_counts'].items():
    print(f"  {col}: {count} missing")
print(f"\n=== Duplicate Rows: {inspection['duplicate_count']} ===")

In [ ]:
# Clean and preprocess
df_clean, dups_removed, missing_before, missing_after = clean_and_preprocess(df_raw)

print(f"Duplicate rows removed: {dups_removed}")
print(f"Missing values before interpolation: {missing_before}")
print(f"Missing values after interpolation: {missing_after}")
print(f"\nProcessed dataset shape: {df_clean.shape}")
print(f"\nNew engineered columns: {[c for c in df_clean.columns if c not in df_raw.columns]}")
print(f"\nFirst 3 rows of cleaned data:")
df_clean.head(3)

In [ ]:
# Chronological Train/Test Split
train_df, test_df = split_train_test(df_clean, train_ratio=0.8)
print(f"Training set: {len(train_df)} rows (Jan 2023 - Oct 2023)")
print(f"Test set:     {len(test_df)} rows (Oct 2023 - Dec 2023)")
print(f"\nThis is a CHRONOLOGICAL split — no future data leaks into training.")

## 6. Exploratory Data Analysis (EDA)

EDA helps us understand the data before modeling. We examine distributions, relationships, and summary statistics.

### 6.1 Summary Statistics

For each continuous variable, we compute:

$$\mu = \frac{1}{N} \sum_{i=1}^{N} x_i \quad \text{(Mean)}$$

$$\sigma^2 = \frac{1}{N-1} \sum_{i=1}^{N} (x_i - \mu)^2 \quad \text{(Sample Variance)}$$

$$\sigma = \sqrt{\sigma^2} \quad \text{(Standard Deviation)}$$

**Quantiles** $Q_p$: The value below which a fraction $p$ of the data falls.
- $Q_{0.25}$ (25th percentile / First Quartile): 25% of observations are below this value
- $Q_{0.50}$ (Median): The middle value
- $Q_{0.75}$ (75th percentile / Third Quartile): 75% of observations are below this value
- **IQR** = $Q_{0.75} - Q_{0.25}$ (Interquartile Range): Spread of the middle 50% of data

In [ ]:
# Compute continuous variable statistics
from src.continuous_analysis import compute_continuous_statistics

stats_df = compute_continuous_statistics(df_clean)
print("=== Continuous Random Variable Summary Statistics ===\n")
display_cols = ["Count", "Mean (mu)", "Std Dev (sigma)", "Min", "25% (Q1)", "50% (Median)",
                "75% (Q3)", "Max", "IQR", "5th Percentile", "95th Percentile", "Skewness"]
print(stats_df[display_cols].to_string())

### 6.2 Visualizations

Let's create the key EDA plots to understand the energy grid data.

In [ ]:
# Plot 1: Demand Time-Series
from src.visualizer import plot_01_demand_timeseries
path = plot_01_demand_timeseries(df_clean)
from IPython.display import Image, display
display(Image(filename=path, width=900))
print("""
INTERPRETATION:
- The top plot shows the full-year hourly demand. Clear seasonal patterns are visible.
- Winter (Jan-Feb) and Summer (Jul-Aug) have HIGHER demand due to heating and cooling loads.
- Spring/Autumn have LOWER demand — the thermal comfort zone requires less HVAC energy.
- The bottom plot zooms into 2 weeks, revealing the DIURNAL cycle: demand peaks in evening (18:00-21:00)
  and drops to minimum at night (02:00-05:00). Weekend drops are also visible.
- The red dashed line shows the mean demand (~24,611 MW) across the year.
""")

In [ ]:
# Plot 2: Renewable Generation
from src.visualizer import plot_02_renewable_generation
path = plot_02_renewable_generation(df_clean)
display(Image(filename=path, width=900))
print("""
INTERPRETATION:
- Solar generation follows a DETERMINISTIC diurnal pattern: zero at night, bell-shaped during day.
  Peak solar occurs around 12:00-13:00 (solar noon). Cloud cover causes stochastic variation.
- Wind generation is highly VOLATILE and STOCHASTIC — it can swing from 0 to 10,000 MW rapidly.
  Wind has no predictable diurnal pattern, making it harder to forecast than solar.
- This contrast explains why solar is more predictable but limited to daytime, while wind provides
  energy 24/7 but with high uncertainty.
""")

In [ ]:
# Plot 3: Demand Histogram
from src.visualizer import plot_03_demand_histogram
path = plot_03_demand_histogram(df_clean)
display(Image(filename=path, width=900))
print("""
INTERPRETATION:
- The histogram shows the EMPIRICAL DISTRIBUTION of electricity demand.
- The distribution is approximately bell-shaped but with heavier tails than a perfect Gaussian.
- The KDE overlay (smooth curve) estimates the probability density function non-parametrically.
- Vertical lines mark the 25th percentile (Q1), Median (50th), and 75th percentile (Q3).
- The IQR (Q3 - Q1) represents the range of the middle 50% of demand observations.
""")

In [ ]:
# Plot 5: Demand vs Temperature Scatter
from src.visualizer import plot_05_demand_vs_temperature
path = plot_05_demand_vs_temperature(df_clean)
display(Image(filename=path, width=900))
print("""
INTERPRETATION:
- This scatter plot reveals the THERMODYNAMIC U-CURVE: demand is high at both extreme cold AND
  extreme hot temperatures.
- Cold temperatures (< 10 C): High demand from space heating (electric heaters, heat pumps)
- Hot temperatures (> 25 C): High demand from air conditioning and cooling systems
- Comfort zone (18-22 C): Minimum demand — no heating or cooling needed
- Color indicates hour of day: bright yellow = midday, dark purple = night
- This U-curve explains why a LINEAR model (degree 1) performs poorly — a POLYNOMIAL (degree 2+)
  is needed to capture the nonlinear relationship.
""")

## 7. Probability Theory — Fundamental Rules

### Concept

Probability theory provides the mathematical framework for reasoning under uncertainty.
In energy forecasting, we use probability to quantify the likelihood of grid events.

### Kolmogorov Axioms (Foundation of Probability)

1. **Non-negativity:** $P(A) \geq 0$ for any event $A$
2. **Normalization:** $P(\Omega) = 1$ (total probability of the sample space is 1)
3. **Additivity:** For mutually exclusive events: $P(A \cup B) = P(A) + P(B)$

### Key Rules

**Joint Probability:** $P(A \cap B)$ — probability that BOTH events occur simultaneously

**Addition Rule (General):**
$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

**Conditional Probability:**
$$P(A|B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0$$

### Energy Grid Events

We define two events based on the dataset:
- **Event A:** Electricity demand is HIGH (above 75th percentile)
- **Event B:** Renewable generation is HIGH (above 75th percentile)

In [ ]:
# Probability Theory Calculations
from src.probability_analysis import compute_probability_events

prob = compute_probability_events(df_clean)

print("=" * 70)
print("PROBABILITY THEORY — FUNDAMENTAL RULES APPLIED TO ENERGY GRID")
print("=" * 70)

print(f"\nEvent A: Demand > {prob['threshold_demand_q75']:.1f} MW (High Demand)")
print(f"Event B: Renewable > {prob['threshold_renewable_q75']:.1f} MW (High Renewable)")

print(f"\n--- Marginal Probabilities ---")
print(f"P(A) = P(High Demand)     = {prob['P(A) [High Demand]']:.4f}")
print(f"P(B) = P(High Renewable)  = {prob['P(B) [High Renewable]']:.4f}")

print(f"\n--- Joint Probability ---")
pa_and_b = prob['P(A \u2229 B) [Joint High Demand & High Renewable]']
print(f"P(A AND B) = {pa_and_b:.4f}")

print(f"\n--- Addition Rule Verification ---")
print(f"P(A OR B) [Empirical]    = {prob['P(A U B) [Empirical Union]']:.4f}")
pa_plus_pb_minus_pab = prob['P(A) + P(B) - P(A \u2229 B) [Theoretical Union]']
print(f"P(A) + P(B) - P(A AND B) = {pa_plus_pb_minus_pab:.4f}")
print(f"Discrepancy              = {prob['Union Rule Discrepancy']:.2e}  (AXIOM VERIFIED!)")

print(f"\n--- Conditional Probabilities ---")
print(f"P(A|B) = P(High Demand | High Renewable) = {prob['P(A|B) [High Demand given High Renewable]']:.4f}")
print(f"P(B|A) = P(High Renewable | High Demand) = {prob['P(B|A) [High Renewable given High Demand]']:.4f}")

print(f"""
INTERPRETATION:
- P(A) = 0.25 means 25% of hours have high demand (by definition of 75th percentile cutoff)
- P(A AND B) = {pa_and_b:.4f} means about {pa_and_b*100:.1f}% of hours have BOTH high demand and high renewable
- The Addition Rule P(A OR B) = P(A) + P(B) - P(A AND B) is verified with zero error
- P(A|B) shows how observing high renewable changes our belief about high demand
""")

## 8. Discrete Random Variables

### Concept

A **Discrete Random Variable** $X$ takes values from a finite or countably infinite set.
Each value has an associated probability.

### Mathematical Framework

**Probability Mass Function (PMF):**
$$P(X = k) = p_k, \quad \sum_{k} p_k = 1$$

**Cumulative Distribution Function (CDF):**
$$F(x) = P(X \leq x) = \sum_{k \leq x} p_k$$

**Expected Value (Mean):**
$$E[X] = \sum_{k} k \cdot P(X = k)$$

**Variance:**
$$\text{Var}(X) = \sum_{k} (k - E[X])^2 \cdot P(X = k)$$

### Application to Energy Grid

We convert the continuous electricity demand into a **3-state discrete random variable:**
- State 0 = Low Demand (below 33rd percentile)
- State 1 = Normal Demand (33rd to 67th percentile)
- State 2 = High Demand (above 67th percentile)

In [ ]:
from src.probability_analysis import analyze_discrete_random_variable

drv = analyze_discrete_random_variable(df_clean)

print("=" * 60)
print("DISCRETE RANDOM VARIABLE ANALYSIS")
print("=" * 60)
print(f"\nCutoffs: Low < {drv['q33_cutoff']:.1f} MW | Normal | High > {drv['q66_cutoff']:.1f} MW")
print(f"\n--- Probability Mass Function (PMF) ---")
for k, v in drv['pmf'].items():
    print(f"  {k}: {v:.4f}")

print(f"\n--- Cumulative Distribution Function (CDF) ---")
for k, v in drv['cdf'].items():
    print(f"  {k}: {v:.4f}")

print(f"\n--- Moments ---")
print(f"  E[X] = {drv['E[X] (Discrete Expectation)']:.4f}")
print(f"  Var(X) = {drv['Var(X) (Discrete Variance)']:.4f}")
print(f"  Std(X) = {drv['Std(X) (Discrete Std Dev)']:.4f}")

print("""
INTERPRETATION:
- Each state has probability ~0.333 because we split at the 33rd and 67th percentiles
- E[X] = 1.0 means the "average state" is Normal Demand (State 1)
- Var(X) = 0.667 quantifies the spread of demand states
- This discretization lets us apply discrete probability theory (PMF, CDF) to continuous energy data
""")

In [ ]:
# Plot 7: PMF Visualization
from src.visualizer import plot_07_discrete_pmf
path = plot_07_discrete_pmf(drv['pmf_series'])
display(Image(filename=path, width=700))

## 9. Bayes' Rule

### Concept

**Bayes' Theorem** is the cornerstone of probabilistic inference. It lets us UPDATE our belief about
an event after observing new evidence.

### Mathematical Formula

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

Where:
- $P(A)$ = **Prior Probability** — our initial belief about event A before observing B
- $P(B|A)$ = **Likelihood** — probability of observing evidence B given that A is true
- $P(B)$ = **Evidence** — total probability of observing B (normalization constant)
- $P(A|B)$ = **Posterior Probability** — updated belief about A after observing B

### Energy Grid Application

**Case 1:** What is the probability of HIGH DEMAND given that we observe HIGH TEMPERATURE?

$$P(\text{High Demand} | \text{High Temp}) = \frac{P(\text{High Temp} | \text{High Demand}) \cdot P(\text{High Demand})}{P(\text{High Temp})}$$

**Case 2:** What is the probability of HIGH DEMAND given LOW RENEWABLE generation?

This is critical for grid operators: when renewables drop, should we prepare for peak demand?

In [ ]:
from src.probability_analysis import compute_bayes_rule

bayes = compute_bayes_rule(df_clean)

print("=" * 70)
print("BAYES' THEOREM — APPLIED TO ENERGY GRID RISK ASSESSMENT")
print("=" * 70)

c1 = bayes['Case 1 (High Temp)']
print(f"\n[CASE 1: P(High Demand | High Temperature)]")
print(f"  Prior P(A) = P(High Demand)              = {c1['Prior P(High Demand)']:.4f}")
print(f"  Likelihood P(B|A) = P(High Temp | High Dem) = {c1['Likelihood P(High Temp | High Demand)']:.4f}")
print(f"  Evidence P(B) = P(High Temp)              = {c1['Evidence P(High Temp)']:.4f}")
print(f"")
print(f"  Bayes Calculation:")
print(f"  P(A|B) = P(B|A) * P(A) / P(B)")
print(f"         = {c1['Likelihood P(High Temp | High Demand)']:.4f} * {c1['Prior P(High Demand)']:.4f} / {c1['Evidence P(High Temp)']:.4f}")
print(f"         = {c1['Posterior P(High Demand | High Temp) [Bayes]']:.4f}")
print(f"  Direct Empirical Check: {c1['Posterior P(High Demand | High Temp) [Direct]']:.4f}")
print(f"  Risk Multiplier (Posterior/Prior): {c1['Risk Ratio (Posterior / Prior)']:.2f}x")

c2 = bayes['Case 2 (Low Renewable)']
print(f"\n[CASE 2: P(High Demand | Low Renewable)]")
print(f"  Prior P(High Demand)                      = {c2['Prior P(High Demand)']:.4f}")
print(f"  Likelihood P(Low Ren | High Demand)       = {c2['Likelihood P(Low Renewable | High Demand)']:.4f}")
print(f"  Evidence P(Low Renewable)                 = {c2['Evidence P(Low Renewable)']:.4f}")
print(f"  Posterior P(High Demand | Low Renewable)   = {c2['Posterior P(High Demand | Low Renewable) [Bayes]']:.4f}")
print(f"  Risk Multiplier: {c2['Risk Ratio (Posterior / Prior)']:.2f}x")

print("""
INTERPRETATION:
Case 1: Observing high temperature INCREASES the probability of high demand from 25% to ~45%.
  This is a 1.79x risk multiplier — the grid operator should prepare more reserves on hot days.
  The posterior matches the direct empirical check, validating our Bayes calculation.

Case 2: Low renewable generation slightly DECREASES high demand probability.
  This seems counterintuitive but makes sense: low renewable often occurs at NIGHT
  when demand is also low. The correlation structure matters!
""")

In [ ]:
# Plot 8: Bayes Visualization
from src.visualizer import plot_08_bayes_probability
path = plot_08_bayes_probability(bayes)
display(Image(filename=path, width=800))

## 10. Independence and Conditional Independence

### Concept

Two events $A$ and $B$ are **statistically independent** if and only if:

$$P(A \cap B) = P(A) \cdot P(B)$$

Equivalently: $P(A|B) = P(A)$ — knowing B does not change our belief about A.

**Conditional Independence:** $A$ and $B$ are conditionally independent given $C$ if:
$$P(A \cap B | C) = P(A|C) \cdot P(B|C)$$

### Important Caveat

> **Correlation is NOT the same as independence.**
> - Zero correlation means no LINEAR relationship, but nonlinear dependence may still exist.
> - Independence implies zero correlation, but zero correlation does NOT imply independence.
> - Example: If $Y = X^2$ and $X \sim N(0,1)$, then $\text{Corr}(X,Y) = 0$ but X and Y are clearly dependent!

In [ ]:
from src.probability_analysis import analyze_independence

ind = analyze_independence(df_clean)

print("=" * 70)
print("INDEPENDENCE ANALYSIS — PAIRWISE EVENT TESTS")
print("=" * 70)

for p in ind['pairwise_tests']:
    print(f"\n  Pair: {p['Event Pair']}")
    print(f"    P(A AND B)   = {p['P(A \u2229 B)']:.4f}")
    print(f"    P(A) * P(B)  = {p['P(A) * P(B)']:.4f}")
    diff = p['|P(A \u2229 B) - P(A)P(B)|']
    print(f"    |Difference| = {diff:.4f}")
    print(f"    Pearson r    = {p['Pearson Correlation (r)']:.4f}")
    verdict = "APPROXIMATELY INDEPENDENT" if p['Statistically Independent?'] else "DEPENDENT"
    print(f"    Verdict: {verdict}")

cond = ind['conditional_test']
print(f"\n--- Conditional Independence Test ---")
print(f"  Condition: {cond['Condition']}")
print(f"  P(Dem AND Sol | H=14)         = {cond['P(Demand \u2229 Solar | Hour=14)']:.4f}")
print(f"  P(Dem|H=14) * P(Sol|H=14)    = {cond['P(Demand|H=14) * P(Solar|H=14)']:.4f}")
print(f"  Discrepancy: {cond['Discrepancy']:.4f}")
ci_verdict = "CONDITIONALLY INDEPENDENT" if cond['Conditionally Independent?'] else "CONDITIONALLY DEPENDENT"
print(f"  Verdict: {ci_verdict}")

print("""
INTERPRETATION:
- Demand & Solar appear approximately independent at the marginal level (small difference),
  but have positive correlation (r=0.34) — both are higher during daytime.
  This shows correlation != independence!
- Demand & Wind are DEPENDENT: high wind slightly reduces probability of high demand.
- Demand & Temperature are DEPENDENT: the thermodynamic U-curve creates strong dependence.
- When we CONDITION on Hour=14 (fixing time of day), Demand and Solar become more
  dependent because at solar noon, high solar irradiance drives both solar output
  and cooling-related demand.
""")

## 11. Continuous Random Variables

### Concept

A **Continuous Random Variable** $X$ can take any value in an interval (uncountably many values).
Unlike discrete RVs, we cannot assign probability to individual points. Instead, probability
is defined over intervals using the **Probability Density Function (PDF)**.

For a continuous RV $X$ with PDF $f(x)$:

$$P(a \leq X \leq b) = \int_a^b f(x) \, dx$$

Key property: $f(x) \geq 0$ and $\int_{-\infty}^{\infty} f(x) \, dx = 1$

### Energy Grid Variables as Continuous RVs

We treat these as continuous random variables:
- **Electricity Demand (MW):** Range ~12,000 to ~40,000 MW
- **Temperature (°C):** Range ~-8 to ~39°C
- **Solar Generation (MW):** Range 0 to ~7,800 MW
- **Wind Generation (MW):** Range 0 to 10,000 MW

In [ ]:
# Detailed continuous statistics
stats_df = compute_continuous_statistics(df_clean)

print("=" * 80)
print("CONTINUOUS RANDOM VARIABLE ANALYSIS")
print("=" * 80)

for var_name in stats_df.index:
    s = stats_df.loc[var_name]
    print(f"\n--- {var_name} ---")
    print(f"  N = {s['Count']:.0f} observations")
    print(f"  Mean (mu)       = {s['Mean (mu)']:.2f}")
    print(f"  Variance (s^2)  = {s['Variance (sigma^2)']:.2f}")
    print(f"  Std Dev (sigma) = {s['Std Dev (sigma)']:.2f}")
    print(f"  Range: [{s['Min']:.2f}, {s['Max']:.2f}]")
    print(f"  Quantiles: Q05={s['5th Percentile']:.1f} | Q25={s['25% (Q1)']:.1f} | Median={s['50% (Median)']:.1f} | Q75={s['75% (Q3)']:.1f} | Q95={s['95th Percentile']:.1f}")
    print(f"  IQR = {s['IQR']:.2f} | Skewness = {s['Skewness']:.3f} | Kurtosis = {s['Kurtosis']:.3f}")

## 12. Mean, Variance, and Quantiles

### Mathematical Definitions

**Sample Mean:** $\bar{x} = \frac{1}{N} \sum_{i=1}^{N} x_i$

**Sample Variance:** $s^2 = \frac{1}{N-1} \sum_{i=1}^{N} (x_i - \bar{x})^2$

(We use $N-1$ for **Bessel's correction** — dividing by $N-1$ gives an unbiased estimate of the population variance)

**Quantile** $Q_p$: The value such that $P(X \leq Q_p) = p$

- $Q_{0.05}$: Only 5% of observations fall below this → used for lower prediction bound
- $Q_{0.95}$: 95% of observations fall below this → used for upper prediction bound
- Together, $[Q_{0.05}, Q_{0.95}]$ forms a **90% prediction interval**

## 13. Probability Density Functions

### Gaussian (Normal) PDF

$$f(x) = \frac{1}{\sigma \sqrt{2\pi}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

### Kernel Density Estimation (KDE)

A **non-parametric** method that doesn't assume any specific distribution shape:

$$\hat{f}(x) = \frac{1}{Nh} \sum_{i=1}^{N} K\left(\frac{x - x_i}{h}\right)$$

where $K$ is a kernel function (typically Gaussian) and $h$ is the bandwidth.

In [ ]:
# Plot 4: PDF Comparison (Gaussian vs KDE)
from src.visualizer import plot_04_probability_density
path = plot_04_probability_density(df_clean)
display(Image(filename=path, width=900))

print("""
INTERPRETATION:
- The PURPLE curve (KDE) shows the actual empirical density — it captures the true shape
  of the demand distribution without assuming normality.
- The RED dashed curve is the fitted Gaussian N(mu, sigma) — it assumes a symmetric bell curve.
- Where the KDE and Gaussian diverge, the data is NON-NORMAL (e.g., heavier tails, slight skew).
- The gray vertical lines show the Gaussian 95% bounds — but since the data isn't perfectly
  Gaussian, the empirical quantile-based bounds are more reliable.
- KEY INSIGHT: f(x) is NOT a probability! It's a density. The probability that demand falls
  in an interval [a,b] is the AREA under the curve between a and b.
""")

## 14. Expectation and Covariance

### Expectation (Expected Value)

For a continuous RV with PDF $f(x)$:
$$E[X] = \int_{-\infty}^{\infty} x \cdot f(x) \, dx$$

In practice with sample data:
$$E[X] \approx \bar{x} = \frac{1}{N} \sum_{i=1}^{N} x_i$$

### Covariance

Measures the **linear co-variation** between two variables:

$$\text{Cov}(X, Y) = \frac{1}{N-1} \sum_{i=1}^{N} (x_i - \bar{x})(y_i - \bar{y})$$

- **Cov > 0:** X and Y tend to increase together (positive relationship)
- **Cov < 0:** When X increases, Y tends to decrease (negative relationship)
- **Cov ≈ 0:** No linear relationship

### Correlation (Normalized Covariance)

$$\text{Corr}(X, Y) = \frac{\text{Cov}(X, Y)}{\sigma_X \cdot \sigma_Y} \in [-1, +1]$$

**Why distinguish covariance from correlation?**
- Covariance depends on the UNITS of X and Y (MW² for demand, MW·°C for demand×temperature)
- Correlation is DIMENSIONLESS and always between -1 and +1, making it easier to compare

In [ ]:
from src.probability_analysis import compute_expectation_and_covariance

exp_cov = compute_expectation_and_covariance(df_clean)

print("=" * 70)
print("EXPECTATION AND COVARIANCE ANALYSIS")
print("=" * 70)

print("\n--- Expected Values E[X] ---")
for var, val in exp_cov['expectations'].items():
    print(f"  E[{var}] = {val:,.2f}")

print("\n--- Covariance with Electricity Demand ---")
for cov_key, cov_val in exp_cov['covariances'].items():
    corr_key = cov_key.replace('Cov', 'Corr')
    corr_val = exp_cov['correlations'][corr_key]
    sign = "POSITIVE" if cov_val > 0 else "NEGATIVE" if cov_val < 0 else "NEAR-ZERO"
    print(f"  {cov_key} = {cov_val:>14,.2f} | {corr_key} = {corr_val:>7.4f} ({sign})")

print("""
INTERPRETATION:
- E[Demand] = ~24,611 MW is the expected (average) hourly electricity demand for the year
- Cov(Solar, Demand) > 0 and Corr = 0.34: Solar and demand are positively related
  (both higher during daytime). This is a moderately strong positive association.
- Cov(Wind, Demand) < 0 and Corr = -0.12: Wind and demand are weakly negatively related
  (wind tends to be slightly higher at night when demand is low).
- Cov(Temperature, Demand) < 0: This seems counterintuitive! But remember the U-curve —
  the linear correlation is slightly negative because the left arm (cold→high demand) dominates.
  The NONLINEAR dependence (U-curve) is strong but not captured by linear covariance!
""")

## 15. Polynomial Curve Fitting

### Concept (PRML Chapter 1 Foundation)

**Polynomial regression** fits a polynomial function of degree $M$ to data:

$$y(x, \mathbf{w}) = w_0 + w_1 x + w_2 x^2 + \cdots + w_M x^M = \sum_{j=0}^{M} w_j x^j$$

We minimize the **sum of squared errors**:

$$E(\mathbf{w}) = \frac{1}{2} \sum_{n=1}^{N} \left( y(x_n, \mathbf{w}) - t_n \right)^2$$

### Error Metrics

$$\text{MAE} = \frac{1}{N} \sum_{i=1}^{N} |y_i - \hat{y}_i|$$

$$\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$$

$$\text{RMSE} = \sqrt{\text{MSE}}$$

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

### Overfitting Warning

As degree $M$ increases, the model fits training data better but may **overfit** — memorizing noise
rather than learning the true pattern. This leads to POOR test performance.
We compare degrees 1, 2, and 3 and watch for this phenomenon.

In [ ]:
from src.models import fit_polynomial_curve

x_train = train_df['temperature_celsius'].values
y_train = train_df['electricity_demand_mw'].values
x_test = test_df['temperature_celsius'].values
y_test = test_df['electricity_demand_mw'].values

poly_results, poly_models, curve_preds = fit_polynomial_curve(x_train, y_train, x_test, y_test)

print("=" * 70)
print("POLYNOMIAL CURVE FITTING: Temperature -> Electricity Demand")
print("=" * 70)
print("\n" + poly_results[['Train RMSE', 'Train R^2', 'Test RMSE', 'Test R^2', 'Test MAE']].to_string())

print("""
INTERPRETATION:
- Degree 1 (Linear): Almost no fit (R^2 ~ 0.001) because the relationship is a U-curve,
  not a straight line. A linear model cannot capture nonlinearity.
- Degree 2 (Quadratic): Much better on training (R^2 = 0.31) as it captures the U-curve.
  But negative test R^2 indicates overfitting to the training period's temperature range.
- Degree 3 (Cubic): Similar performance to degree 2.
- KEY LESSON: Temperature alone explains only ~31% of demand variance. Other features
  (hour, day, solar, wind) are needed for a good model — motivating the multivariate
  supervised learning approach in Section 16.
""")

In [ ]:
# Plot 6: Polynomial Curves
from src.visualizer import plot_06_polynomial_regression_curve
path = plot_06_polynomial_regression_curve(curve_preds, poly_results)
display(Image(filename=path, width=900))

## 16. Supervised Learning Model

### Concept

In **Supervised Learning**, we have labeled training data $\{(\mathbf{x}_i, y_i)\}_{i=1}^{N}$ and learn
a function $f(\mathbf{x}) \rightarrow y$ that maps input features to the target variable.

### Our Approach

We compare two models:

**Model 1: Baseline Linear Regression**
$$\hat{y} = w_0 + w_1 \cdot \text{temp} + w_2 \cdot \text{solar} + w_3 \cdot \text{wind} + w_4 \cdot \text{hour} + \cdots$$

**Model 2: Feature-Engineered Polynomial Regression**

We add domain-guided features:
- $\text{temp}^2$ — captures the thermodynamic U-curve
- $\sin(2\pi \cdot \text{hour}/24)$, $\cos(2\pi \cdot \text{hour}/24)$ — captures cyclic diurnal pattern

This is still linear regression but with engineered features — demonstrating how domain knowledge improves models.

In [ ]:
from src.models import train_supervised_model

sup = train_supervised_model(train_df, test_df)

print("=" * 70)
print("SUPERVISED LEARNING: MODEL COMPARISON")
print("=" * 70)
print("\n" + sup['comparison_df'].to_string())

print("""
INTERPRETATION:
- Linear Regression (Base): Test R^2 is negative! The model performs worse than predicting
  the training mean. This happens because the model overfits to training data patterns that
  don't transfer to the test period (different season).

- Feature-Engineered Model: Test R^2 = 0.809! Adding temp^2 (U-curve) and hour sin/cos
  (diurnal cycle) dramatically improves predictions. RMSE drops from ~4500 to ~1570 MW.

- KEY INSIGHT: Good features matter more than complex algorithms at the Unit 1 level.
  Understanding the PHYSICS of the problem (U-curve, diurnal cycle) leads to better models
  than blindly throwing data at a linear regression.
""")

In [ ]:
# Plot 10: Actual vs Predicted Demand
from src.visualizer import plot_10_actual_vs_predicted_demand
path = plot_10_actual_vs_predicted_demand(sup['y_test'], sup['y_pred_test'])
display(Image(filename=path, width=900))

## 17. Unsupervised Learning — K-Means Clustering

### Concept

In **Unsupervised Learning**, there are no labels. The algorithm discovers hidden patterns or structure
in the data on its own.

### K-Means Algorithm

1. Initialize $K$ cluster centroids randomly
2. **Assignment step:** Assign each data point to its nearest centroid
3. **Update step:** Recompute each centroid as the mean of assigned points
4. Repeat until convergence

The objective is to minimize:
$$J = \sum_{k=1}^{K} \sum_{i \in C_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$$

### Energy Grid Application

We cluster hours using: Demand, Solar, Wind, Temperature → discover operational regimes.

In [ ]:
from src.models import train_unsupervised_kmeans

kmeans_res = train_unsupervised_kmeans(df_clean, n_clusters=4)

print("=" * 70)
print("UNSUPERVISED LEARNING: K-MEANS CLUSTERING (K=4)")
print("=" * 70)
print("\nCluster Centroids & Regime Descriptions:\n")
print(kmeans_res['cluster_summary'].to_string())

print("""
INTERPRETATION:
The algorithm automatically discovered 4 meaningful grid operating regimes:

1. WINTER HEATING PEAK (~37%): Cold temperatures, high demand from space heating,
   low solar (short winter days). Grid relies on conventional generation.

2. OFF-PEAK BASELOAD (~27%): Nighttime hours with low demand and low renewables.
   Conventional baseload plants (nuclear, gas) dominate.

3. SUMMER COOLING PEAK (~26%): Hot temperatures driving air conditioning demand,
   high solar generation helps offset some demand.

4. WINDY TRANSITION (~10%): Periods of high wind generation with moderate demand.
   These are valuable for renewable energy integration.
""")

In [ ]:
# Plot 9: Clustering Visualization
from src.visualizer import plot_09_clustering_visualization
path = plot_09_clustering_visualization(df_clean, kmeans_res)
display(Image(filename=path, width=900))

## 18. Uncertainty Quantification — The Most Important Section

### Why Uncertainty Matters

A point prediction ("demand will be 25,000 MW") tells an incomplete story. Grid operators need:

1. **How confident is this prediction?**
2. **What is the worst-case scenario?**
3. **Should we activate reserve generators?**

### Methodology

1. Compute **prediction residuals** (errors) on the test set:
$$\varepsilon_i = y_i^{\text{actual}} - y_i^{\text{predicted}}$$

2. Analyze the **distribution of errors**: mean, variance, quantiles

3. Construct **prediction intervals** using error quantiles:
$$\text{Lower Bound} = \hat{y} + Q_{0.05}(\varepsilon) \quad \text{(5th percentile of errors)}$$
$$\text{Upper Bound} = \hat{y} + Q_{0.95}(\varepsilon) \quad \text{(95th percentile of errors)}$$

This gives a **90% prediction interval**: we expect ~90% of actual values to fall within this range.

### Alternative: Gaussian Parametric Interval
$$\hat{y} \pm 1.96 \cdot \sigma_\varepsilon \quad \text{(95% interval assuming Gaussian errors)}$$

> **Important:** The prediction interval represents UNCERTAINTY, not a guarantee.
> Even a 90% interval means ~10% of actual values will fall outside the bounds.

In [ ]:
from src.uncertainty import analyze_prediction_uncertainty

uq_stats, pred_df, residuals = analyze_prediction_uncertainty(
    sup['y_test'], sup['y_pred_test'], alpha=0.10
)

print("=" * 70)
print("UNCERTAINTY QUANTIFICATION RESULTS")
print("=" * 70)
print("\n--- Residual Distribution Statistics ---")
for k, v in uq_stats.items():
    print(f"  {k}: {v:.2f}")

print("""
INTERPRETATION:
- Mean Error = -244 MW: The model slightly OVER-predicts demand on average (small bias)
- Std Dev = 1,551 MW: Typical prediction error is about +/- 1,551 MW
- 90% Empirical Interval Width = ~4,835 MW: We are 90% confident actual demand falls
  within ~4,835 MW of our prediction
- Empirical 90% Coverage = 89.95%: Almost exactly 90% of actual values fall within our
  90% interval — this validates our uncertainty quantification method!
- Gaussian 95% Coverage = 97.32%: The Gaussian interval is wider and over-covers,
  suggesting the error distribution has lighter tails than a Gaussian.
""")

In [ ]:
# Plot 11: Error Distribution
from src.visualizer import plot_11_prediction_error_distribution
path = plot_11_prediction_error_distribution(residuals, uq_stats)
display(Image(filename=path, width=900))

In [ ]:
# Plot 12: Prediction Interval Visualization
from src.visualizer import plot_12_prediction_interval_uncertainty
path = plot_12_prediction_interval_uncertainty(pred_df)
display(Image(filename=path, width=900))
print("""
INTERPRETATION:
- Blue line: Point forecast from our supervised regression model
- Red dots: Actual observed demand values
- Blue shaded band: 90% prediction interval [Lower Bound, Upper Bound]
- Most red dots fall INSIDE the blue band (~90%), confirming our uncertainty is well-calibrated
- The interval widens during volatile periods and narrows during stable periods
- Grid operators use this band to plan reserve generation capacity
""")

## 19. Results and Visualization Summary

### All 12 Required Visualizations

| # | Visualization | Unit 1 Topic |
|---|---------------|-------------|
| 1 | Demand time-series | Data exploration, time series |
| 2 | Renewable generation | Data exploration, solar vs wind |
| 3 | Demand histogram | Probability distribution, quantiles |
| 4 | Probability density (KDE vs Gaussian) | Continuous RV, PDF |
| 5 | Demand vs Temperature scatter | Correlation, nonlinear relationships |
| 6 | Polynomial regression curves | Polynomial curve fitting, overfitting |
| 7 | Discrete PMF | Discrete random variables |
| 8 | Bayes' probability | Bayes' Rule |
| 9 | K-Means clustering | Unsupervised learning |
| 10 | Actual vs Predicted demand | Supervised learning |
| 11 | Prediction error distribution | Uncertainty, error quantiles |
| 12 | Prediction interval | Uncertainty quantification |

### Key Numerical Results

| Metric | Value |
|--------|-------|
| Feature-Engineered Model Test R² | 0.809 |
| Test RMSE | ~1,570 MW |
| 90% Prediction Interval Coverage | 89.95% |
| Bayes P(High Demand \| High Temp) | 44.77% (vs 25% prior) |
| K-Means discovered 4 regimes | Winter, Summer, Night, Windy |
| Addition Rule verified | 0 discrepancy |

## 20. Conclusion

This project demonstrates that **Unit 1 ML and Probability Theory concepts** are sufficient to build
a practical, interpretable energy demand forecasting system with uncertainty quantification.

### Key Takeaways:

1. **Machine Learning is Data-Driven:** Instead of hardcoding rules about energy demand, we let the
   model LEARN patterns from 8,760 hours of historical data.

2. **Feature Engineering > Complex Algorithms:** Adding physics-inspired features (temp², hour sin/cos)
   improved R² from negative to 0.809 — no neural networks needed.

3. **Probability Theory Enables Uncertainty:** Using Bayes' Rule, we can update risk assessments
   dynamically. Using error quantiles, we provide calibrated prediction intervals.

4. **Unsupervised Learning Reveals Hidden Patterns:** K-Means discovered 4 meaningful grid regimes
   without any labels — validating the concept of pattern discovery.

5. **Independence ≠ Zero Correlation:** The analysis of demand vs solar showed that marginal
   independence can coexist with positive correlation due to confounding variables (time of day).

6. **Polynomial Fitting Demonstrates the Bias-Variance Tradeoff:** Degree 1 underfits (high bias),
   while very high degrees would overfit (high variance). Degree 2 captures the U-curve.

## 21. Future Scope

1. **Time Series Models:** ARIMA, Prophet for temporal autocorrelation
2. **Advanced ML:** Gradient Boosting (XGBoost), Random Forests for better accuracy
3. **Deep Learning:** LSTM, Transformer architectures for sequence modeling
4. **Probabilistic Models:** Bayesian Neural Networks, Gaussian Processes for native uncertainty
5. **Real-Time Dashboard:** Web-based interface (Flask/Streamlit) for live monitoring
6. **Real Dataset:** Apply to actual grid data (ENTSOE, PJM, AEMO)
7. **Multi-Step Forecasting:** Predict demand 24-48 hours ahead
8. **Demand Response Integration:** Include price signals and consumer behavior

---

*This project was built entirely within Unit 1 scope — no deep learning, no advanced algorithms.
Every technique can be fully explained and defended in a viva examination.*